# Feature Engineering

In [317]:
import pandas as pd
import numpy as np
import sqlite3

conn = sqlite3.connect('../diabetes-readmission.db')

df = pd.read_sql("""
    SELECT * FROM encounters
""", conn)

df_clone = df
df_clone.head()

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


## Reasoning for later

Start by dropping the columns I know I won't be needing

In [318]:
columns_to_drop = ['weight', 'medical_specialty', 'payer_code', 'max_glu_serum', 'encounter_id', 'change', 'diabetesMed']

df_clone.drop(columns=columns_to_drop, inplace=True)
df_clone.shape

(101766, 43)

## Reasoning For Later

Deduplicate the encounters to unique patients only. This will reduce the memory held as well as reduce time for computations later.

In [319]:
df_clone.drop_duplicates(subset='patient_nbr', keep='first', inplace=True)
df_clone.drop(columns='patient_nbr', inplace=True)
df_clone = df_clone.reset_index(drop=True)
df_clone.shape

(71518, 42)

In [320]:
all_med_cols = ['metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
                'glimepiride', 'acetohexamide', 'glipizide', 'glyburide',
                'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose',
                'miglitol', 'troglitazone', 'tolazamide', 'examide',
                'citoglipton', 'insulin', 'glyburide-metformin',
                'glipizide-metformin', 'glimepiride-pioglitazone',
                'metformin-rosiglitazone', 'metformin-pioglitazone']

# Create binary change indicator before dropping
df_clone['any_med_change'] = (df[all_med_cols].isin(['Up', 'Down'])).any(axis=1).astype(int)

# Drop all individual medication columns
df_clone.drop(columns=all_med_cols, inplace=True)
df_clone.shape

(71518, 20)

In [321]:
print(df_clone.select_dtypes(include='object').columns.tolist())

['race', 'gender', 'age', 'diag_1', 'diag_2', 'diag_3', 'A1Cresult', 'readmitted']


In [322]:
columns = df_clone.select_dtypes(include='object').columns.tolist()

for col in columns:
    print(f"\nMissing Values for {col}: {df_clone[col].value_counts()}")


Missing Values for race: race
Caucasian          53491
AfricanAmerican    12887
?                   1948
Hispanic            1517
Other               1178
Asian                497
Name: count, dtype: int64

Missing Values for gender: gender
Female             38025
Male               33490
Unknown/Invalid        3
Name: count, dtype: int64

Missing Values for age: age
[70-80)     18210
[60-70)     15960
[50-60)     12466
[80-90)     11589
[40-50)      6878
[30-40)      2699
[90-100)     1900
[20-30)      1127
[10-20)       535
[0-10)        154
Name: count, dtype: int64

Missing Values for diag_1: diag_1
414       5233
428       3980
786       3040
410       2902
486       2439
          ... 
848          1
250.51       1
691          1
834          1
V51          1
Name: count, Length: 697, dtype: int64

Missing Values for diag_2: diag_2
250     5009
276     4604
428     4335
427     3539
401     3088
        ... 
704        1
911        1
520        1
E890       1
927        1
Name:

### Reasoning For Later

Modify the columns I kept that have missing values

In [323]:
def replace_missing_data(entry):
    entry = str(entry).strip()
    if entry == "?":
        return "Unknown"
    return entry

def categorize_icd9(code):
    code = str(code).strip()
    
    if code == '?':
        return "Unknown"
    
    if code.startswith('V'):
        return "Supplementary"
    if code.startswith("E"):
        return "External"
    
    try:
        num = float(code)
    except ValueError:
        return "Other"
    
    if num < 140:
        return "Infectious Disease"
    elif num >=140 and num < 240:
        return "Neoplasms"
    elif num >=240 and num < 280:
        if num >= 250 and num < 251:
            return "Diabetes"
        return "Endocrine"
    elif num >=280 and num < 290:
        return "Blood Disease"
    elif num >=290 and num < 320:
        return "Behavioral"
    elif num >=320 and num < 390:
        return "Nervous System"
    elif num >=390 and num < 460:
        return "Circulatory"
    elif num >=460 and num < 520:
        return "Respiratory"
    elif num >=520 and num < 580:
        return "Digestive"
    elif num >=580 and num < 630:
        return "Genitourinary"
    elif num >=630 and num < 680:
        return "Pregnancy"
    elif num >=680 and num < 710:
        return "Skin"
    elif num >=710 and num < 740:
        return "Musculoskeletal"
    elif num >=740 and num < 760:
        return "Congenital Anomalies"
    elif num >=760 and num < 780:
        return "Perinatal"
    elif num >=780 and num < 800:
        return "Ill-defined Conditions"
    elif num >=800 and num < 1000:
        return "Injury and Poisoning"
    
    

    return "Other"


In [324]:
df_clone['race'] = df['race'].apply(replace_missing_data)
df_clone['diag_1'] = df['diag_1'].apply(categorize_icd9)
df_clone['diag_2'] = df['diag_2'].apply(categorize_icd9)
df_clone['diag_3'] = df['diag_3'].apply(categorize_icd9)

print(df_clone['diag_1'].value_counts())
print(df_clone['race'].value_counts(normalize=True))
df_clone.shape

diag_1
Circulatory               16449
Respiratory                4769
Digestive                  4671
Diabetes                   4457
Ill-defined Conditions     4172
Injury and Poisoning       3448
Musculoskeletal            2955
Genitourinary              2370
Neoplasms                  1998
Endocrine                  1424
Skin                       1277
Infectious Disease         1232
Behavioral                 1186
Supplementary               760
Nervous System              600
Blood Disease               461
Pregnancy                   406
Congenital Anomalies         31
Unknown                       9
Name: count, dtype: int64
race
Caucasian          0.738548
AfricanAmerican    0.200513
Unknown            0.023028
Hispanic           0.018643
Other              0.013802
Asian              0.005467
Name: proportion, dtype: float64


(71518, 20)

In [325]:
print(df_clone.select_dtypes(include='object').columns.tolist())
df_clone.head()

['race', 'gender', 'age', 'diag_1', 'diag_2', 'diag_3', 'A1Cresult', 'readmitted']


,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,A1Cresult,readmitted,any_med_change
0,Caucasian,Female,[0-10),6,25,1,1,41,0,1,0,0,0,Diabetes,Unknown,Unknown,1,None,NO,0.0
1,Caucasian,Female,[10-20),1,1,7,3,59,0,18,0,0,0,Endocrine,Diabetes,Endocrine,9,None,>30,1.0
2,AfricanAmerican,Female,[20-30),1,1,7,2,11,5,13,2,0,1,Pregnancy,Diabetes,Supplementary,6,None,NO,0.0
3,Caucasian,Male,[30-40),1,1,7,2,44,1,16,0,0,0,Infectious Disease,Diabetes,Circulatory,7,None,NO,1.0
4,Caucasian,Male,[40-50),1,1,7,1,51,0,8,0,0,0,Neoplasms,Neoplasms,Diabetes,5,None,NO,0.0


In [326]:
df_clone = pd.get_dummies(df_clone, columns=['age', 'race', 'gender', 'diag_1', 'diag_2', 'diag_3', 'A1Cresult'], drop_first=True)
df_clone.shape
df_clone.head()

,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,...,diag_3_Musculoskeletal,diag_3_Neoplasms,diag_3_Nervous System,diag_3_Pregnancy,diag_3_Respiratory,diag_3_Skin,diag_3_Supplementary,diag_3_Unknown,A1Cresult_>8,A1Cresult_Norm
0,6,25,1,1,41,0,1,0,0,0,...,False,False,False,False,False,False,False,True,False,False
1,1,1,7,3,59,0,18,0,0,0,...,False,False,False,False,False,False,False,False,False,False
2,1,1,7,2,11,5,13,2,0,1,...,False,False,False,False,False,False,True,False,False,False
3,1,1,7,2,44,1,16,0,0,0,...,False,False,False,False,False,False,False,False,False,False
4,1,1,7,1,51,0,8,0,0,0,...,False,False,False,False,False,False,False,False,False,False


In [327]:
codes = [11, 13, 14, 19, 20, 21]
df_clone = df_clone[~df_clone['discharge_disposition_id'].isin(codes)]
df_clone.drop(columns=['admission_type_id', 'discharge_disposition_id', 'admission_source_id'], inplace=True)
df_clone.head()

,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses,readmitted,any_med_change,...,diag_3_Musculoskeletal,diag_3_Neoplasms,diag_3_Nervous System,diag_3_Pregnancy,diag_3_Respiratory,diag_3_Skin,diag_3_Supplementary,diag_3_Unknown,A1Cresult_>8,A1Cresult_Norm
0,1,41,0,1,0,0,0,1,NO,0.0,...,False,False,False,False,False,False,False,True,False,False
1,3,59,0,18,0,0,0,9,>30,1.0,...,False,False,False,False,False,False,False,False,False,False
2,2,11,5,13,2,0,1,6,NO,0.0,...,False,False,False,False,False,False,True,False,False,False
3,2,44,1,16,0,0,0,7,NO,1.0,...,False,False,False,False,False,False,False,False,False,False
4,1,51,0,8,0,0,0,5,NO,0.0,...,False,False,False,False,False,False,False,False,False,False


In [328]:
print(df_clone.columns.tolist())
print(df_clone.shape)

['time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses', 'readmitted', 'any_med_change', 'age_[10-20)', 'age_[20-30)', 'age_[30-40)', 'age_[40-50)', 'age_[50-60)', 'age_[60-70)', 'age_[70-80)', 'age_[80-90)', 'age_[90-100)', 'race_Asian', 'race_Caucasian', 'race_Hispanic', 'race_Other', 'race_Unknown', 'gender_Male', 'gender_Unknown/Invalid', 'diag_1_Blood Disease', 'diag_1_Circulatory', 'diag_1_Congenital Anomalies', 'diag_1_Diabetes', 'diag_1_Digestive', 'diag_1_Endocrine', 'diag_1_Genitourinary', 'diag_1_Ill-defined Conditions', 'diag_1_Infectious Disease', 'diag_1_Injury and Poisoning', 'diag_1_Musculoskeletal', 'diag_1_Neoplasms', 'diag_1_Nervous System', 'diag_1_Pregnancy', 'diag_1_Respiratory', 'diag_1_Skin', 'diag_1_Supplementary', 'diag_1_Unknown', 'diag_2_Blood Disease', 'diag_2_Circulatory', 'diag_2_Congenital Anomalies', 'diag_2_Diabetes', 'diag_2_Digestive', 'diag_2_Endo

In [329]:
df_clone['readmitted'] = (df_clone['readmitted'] != 'NO').astype(int)

print(df_clone['readmitted'].value_counts())

readmitted
0    41474
1    28499
Name: count, dtype: int64
